# Week 4 — Day 2: Cross-Validation

In Day 1 I evaluated the models using one train/validation/test split.  
The Logistic Regression validation F1 was about `0.6567`.

In this notebook I want to check whether that result stays similar when the validation data changes. I keep the same COVID-19 dataset and the same modeling setup, then use 5-fold cross-validation on the development data.

I also keep the final test set outside the folds.

## 1. Imports

I use the same basic libraries from Day 1, with three new tools for cross-validation:

- `KFold`
- `StratifiedKFold`
- `cross_val_score`

I keep F1-score as the main metric because the positive class is much smaller than the negative class.

In [1]:
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

## 2. Load the Same Dataset

I use the same COVID-19 dataset from Week 3 and Day 1.  
Keeping the dataset fixed makes the comparison easier because I only want to change the evaluation method.

In [2]:
DATA_PATH = "../../week-3/corona dataset/corona_tested_individuals_ver_006.english.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (278848, 10)


,test_date,cough,fever,sore_throat,shortness_of_breath,head_ache,corona_result,age_60_and_above,gender,test_indication
0,2020-04-30,0.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
1,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
2,2020-04-30,0.0,1.0,0.0,0.0,0.0,negative,NaN,male,Other
3,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
4,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,male,Other


## 3. Prepare the Same Modeling Data

I keep the same features and target preparation from Day 1:

- the five symptom columns,
- `contact_with_confirmed`,
- `abroad`,
- negative cases as `0`,
- positive cases as `1`.

I do not change the feature set here because I want the comparison with Day 1 to stay fair.

In [3]:
symptom_features = [
    "cough",
    "fever",
    "sore_throat",
    "shortness_of_breath",
    "head_ache",
]

target_column = "corona_result"

model_df = df[
    df[target_column].isin(["negative", "positive"])
].copy()

model_df = model_df.dropna(subset=symptom_features)

model_df["contact_with_confirmed"] = (
    model_df["test_indication"] == "Contact with confirmed"
).astype(int)

model_df["abroad"] = (
    model_df["test_indication"] == "Abroad"
).astype(int)

feature_columns = symptom_features + [
    "contact_with_confirmed",
    "abroad",
]

X = model_df[feature_columns].astype(int)

y = (
    model_df[target_column]
    .map({"negative": 0, "positive": 1})
    .astype(int)
)

print("Modeling samples:", len(X))
print("Number of features:", X.shape[1])
print("\nTarget counts:")
print(y.value_counts())

print("\nPositive-class rate:")
print(f"{y.mean() * 100:.2f}%")

Modeling samples: 274702
Number of features: 7

Target counts:
corona_result
0    260008
1     14694
Name: count, dtype: int64

Positive-class rate:
5.35%


### Target Distribution

The positive class is only about 5% of the data, so this is still an imbalanced classification problem.

That is why I continue using F1-score as the main metric. It is also the reason I want the folds to keep a similar positive/negative ratio instead of letting the class balance change randomly between folds.

## 4. Recreate the Day 1 Development/Test Split

I recreate the same 80/20 development/test split from Day 1 using the same `random_state` and stratification.

Cross-validation will use only the development set. The held-out test set stays separate.

In [4]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Development samples:", len(X_dev))
print("Final test samples:", len(X_test))
print(f"Development positive rate: {y_dev.mean() * 100:.2f}%")
print(f"Test positive rate: {y_test.mean() * 100:.2f}%")

Development samples: 219761
Final test samples: 54941
Development positive rate: 5.35%
Test positive rate: 5.35%


The split is now:

```text
Full modeling data
│
├── 80% Development  → used for cross-validation
└── 20% Test         → kept outside the folds
```

The test set does not contribute to the fold scores, the CV mean, or the CV standard deviation.

## 5. Reproduce the Day 1 Single Validation Result

Before running cross-validation, I recreate yesterday's train/validation split so I have the original result to compare against.

Inside the 80% development data, the split gives:

- 60% of the full data for training,
- 20% for validation.

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev,
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

Training samples: 164820
Validation samples: 54941


## 6. Reuse Logistic Regression from Day 1

Logistic Regression had the best default validation F1 among the model families I compared yesterday, so I use the same model configuration here.

For the first CV experiment I keep its normal `.predict()` behavior. This makes the comparison directly match the Day 1 default Logistic Regression result.

Yesterday I also tested a threshold of `0.20`. I will come back to that after the main CV result instead of mixing both questions at once.

In [6]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

model.fit(X_train, y_train)

single_val_predictions = model.predict(X_val)
single_split_f1 = f1_score(y_val, single_val_predictions)

print(f"Day 1 single-split Validation F1: {single_split_f1:.4f}")

Day 1 single-split Validation F1: 0.6567


The recreated single-validation result is about:

```text
F1 = 0.6567
```

This score is correct for that validation split, but by itself it does not show whether the result would stay similar with a different validation sample.

## 7. 5-Fold Cross-Validation

I split the development set into five folds.

In every round, four folds are used for training and one fold is used for validation. The validation fold changes each time.

```text
Round 1: 4 folds train | Fold 1 validates
Round 2: 4 folds train | Fold 2 validates
Round 3: 4 folds train | Fold 3 validates
Round 4: 4 folds train | Fold 4 validates
Round 5: 4 folds train | Fold 5 validates
```

Each round fits a fresh model. A sample can therefore be validation data in one round and training data in another without carrying model state between the rounds.

For each model being evaluated, its own validation fold was not part of that model's training data.

## 8. K-Fold vs. Stratified K-Fold

Both methods rotate the validation fold.

The difference is that `KFold` only splits the rows, while `StratifiedKFold` also tries to preserve the target class proportions.

Because only about 5% of this dataset is positive, I prefer Stratified K-Fold for the model evaluation.

In [7]:
plain_kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

I first compare the positive-class percentage in the validation folds from normal K-Fold and Stratified K-Fold.

This cell only inspects the folds; it does not train a model.

In [8]:
fold_balance_rows = []

plain_splits = plain_kfold.split(X_dev)
stratified_splits = stratified_kfold.split(X_dev, y_dev)

for fold_number, (plain_split, stratified_split) in enumerate(
    zip(plain_splits, stratified_splits),
    start=1,
):
    _, plain_val_idx = plain_split
    _, stratified_val_idx = stratified_split

    fold_balance_rows.append({
        "Fold": fold_number,
        "KFold Positive Rate (%)": (
            y_dev.iloc[plain_val_idx].mean() * 100
        ),
        "StratifiedKFold Positive Rate (%)": (
            y_dev.iloc[stratified_val_idx].mean() * 100
        ),
    })

fold_balance_df = pd.DataFrame(fold_balance_rows)

fold_balance_df.round(3)

,Fold,KFold Positive Rate (%),StratifiedKFold Positive Rate (%)
0,1,5.331,5.349
1,2,5.258,5.349
2,3,5.479,5.349
3,4,5.253,5.349
4,5,5.424,5.349


The ordinary K-Fold percentages are already fairly close because this dataset is large, but they still move from fold to fold.

With Stratified K-Fold, the positive rate stays essentially the same in every fold. That is the behavior I want for this imbalanced target.

## 9. Inspect the Stratified Folds

I also check the fold sizes and the class ratio before training.

With 5 folds, each round uses about 80% of the development set for training and 20% for validation. These percentages are inside the development set, not the full dataset.

In [9]:
fold_rows = []

for fold_number, (train_idx, val_idx) in enumerate(
    stratified_kfold.split(X_dev, y_dev),
    start=1,
):
    y_fold_train = y_dev.iloc[train_idx]
    y_fold_val = y_dev.iloc[val_idx]

    fold_rows.append({
        "Fold": fold_number,
        "Training Samples": len(train_idx),
        "Validation Samples": len(val_idx),
        "Training Positive Rate (%)": y_fold_train.mean() * 100,
        "Validation Positive Rate (%)": y_fold_val.mean() * 100,
    })

fold_summary_df = pd.DataFrame(fold_rows)

fold_summary_df.round(3)

,Fold,Training Samples,Validation Samples,Training Positive Rate (%),Validation Positive Rate (%)
0,1,175808,43953,5.349,5.349
1,2,175809,43952,5.349,5.349
2,3,175809,43952,5.349,5.349
3,4,175809,43952,5.349,5.349
4,5,175809,43952,5.349,5.349


## 10. Run 5-Fold Cross-Validation

Now I evaluate the same Logistic Regression configuration across the five stratified folds.

`cross_val_score` fits a fresh copy of the estimator in every round and returns one F1-score for each validation fold.

In [10]:
cv_scores = cross_val_score(
    model,
    X_dev,
    y_dev,
    cv=stratified_kfold,
    scoring="f1",
)

cv_scores

array([0.65360392, 0.66574713, 0.66171349, 0.65861027, 0.64959631])

## 11. Fold Scores

I look at the five scores before calculating the average. This makes it easier to see whether one fold behaves very differently from the others.

In [11]:
fold_scores_df = pd.DataFrame({
    "Fold": range(1, 6),
    "F1": cv_scores,
})

fold_scores_df.round(4)

,Fold,F1
0,1,0.6536
1,2,0.6657
2,3,0.6617
3,4,0.6586
4,5,0.6496


The five scores are quite close to each other. The lowest is about `0.6496` and the highest is about `0.6657`.

This is useful because one validation split could have hidden that variation. Cross-validation gives me a clearer picture of how stable the result is.

## 12. Mean CV Score

The mean gives one summary of the performance across all five validation folds.

I use the mean rather than the best fold because choosing the best fold would again make the conclusion depend on one favorable split.

In [12]:
cv_mean = cv_scores.mean()

print(f"Mean CV F1: {cv_mean:.4f}")

Mean CV F1: 0.6579


## 13. Standard Deviation

The standard deviation shows how much the five fold scores move around their mean.

A small value means the scores stayed close together. A larger value would tell me that the result is more sensitive to which samples fall into training and validation.

In [13]:
cv_std = cv_scores.std()

print(f"CV F1: {cv_mean:.4f} ± {cv_std:.4f}")

CV F1: 0.6579 ± 0.0057


### Interpreting My CV Result

The mean F1 is `0.6579` and the standard deviation is `0.0057`.

For this experiment, the small standard deviation matches what I saw in the individual fold scores: the results are fairly stable across the five splits.

The standard deviation does not explain *why* a model is unstable if the value is large. It only shows that the performance is changing more between folds, which would need further investigation.

## 14. Compare with Day 1

Now I can compare yesterday's single validation result with the cross-validation estimate.

In [14]:
evaluation_comparison = pd.DataFrame({
    "Evaluation Method": [
        "Day 1 — Single Validation Split",
        "Day 2 — 5-Fold Cross-Validation",
    ],
    "F1 Estimate": [
        single_split_f1,
        cv_mean,
    ],
    "Score Std": [
        None,
        cv_std,
    ],
})

evaluation_comparison.round(4)

,Evaluation Method,F1 Estimate,Score Std
0,Day 1 — Single Validation Split,0.6567,NaN
1,Day 2 — 5-Fold Cross-Validation,0.6579,0.0057


### Comparison

Day 1 gave a single validation F1 of about `0.6567`.

The 5-fold CV mean is about `0.6579` with a standard deviation of `0.0057`.

The difference between `0.6567` and `0.6579` is very small, so I do not treat this as a real model improvement. Instead, it gives me more confidence that yesterday's validation score was reasonably representative and was not just the result of one unusually easy or difficult split.

## 15. Why I Do Not Keep the Best Fold Model

The five fitted models are temporary models used to evaluate the same setup on different splits.

I do not keep the model from the fold with the highest F1. The useful result is the overall behavior across the folds, not the luckiest fitted model.

After the model configuration is fixed, a new final model can be fitted using all available development data.

## 16. Check Yesterday's Threshold Across the Folds

Yesterday, lowering the Logistic Regression threshold to `0.20` improved the single-validation F1 from about `0.6567` to `0.6665`.

I first check that same `0.20` threshold across the five stratified folds. If the improvement is still present across the folds, I then try a small range of nearby thresholds to see whether `0.20` is already close to the best region.

In [15]:
threshold = 0.20
threshold_cv_scores = []

for train_idx, val_idx in stratified_kfold.split(X_dev, y_dev):
    X_fold_train = X_dev.iloc[train_idx]
    X_fold_val = X_dev.iloc[val_idx]
    y_fold_train = y_dev.iloc[train_idx]
    y_fold_val = y_dev.iloc[val_idx]

    fold_model = LogisticRegression(
        max_iter=1000,
        random_state=42,
    )

    fold_model.fit(X_fold_train, y_fold_train)

    fold_probability = fold_model.predict_proba(X_fold_val)[:, 1]
    fold_predictions = (fold_probability >= threshold).astype(int)

    threshold_cv_scores.append(
        f1_score(y_fold_val, fold_predictions)
    )

threshold_cv_scores = pd.Series(
    threshold_cv_scores,
    index=range(1, 6),
    name="F1",
)

threshold_cv_scores

1    0.661651
2    0.675371
3    0.670743
4    0.670292
5    0.659301
Name: F1, dtype: float64

In [16]:
threshold_cv_mean = threshold_cv_scores.mean()
threshold_cv_std = threshold_cv_scores.std(ddof=0)

threshold_comparison = pd.DataFrame({
    "Configuration": [
        "Default Logistic Regression",
        "Logistic Regression (threshold=0.20)",
    ],
    "Mean CV F1": [
        cv_mean,
        threshold_cv_mean,
    ],
    "CV Std": [
        cv_std,
        threshold_cv_std,
    ],
})

threshold_comparison.round(4)

,Configuration,Mean CV F1,CV Std
0,Default Logistic Regression,0.6579,0.0057
1,Logistic Regression (threshold=0.20),0.6675,0.0060


In [17]:
cv_difference = threshold_cv_mean - cv_mean

if cv_difference > 0:
    print(
        f"Threshold 0.20 improved mean CV F1 by {cv_difference:.4f}."
    )
else:
    print(
        f"Threshold 0.20 did not improve mean CV F1 "
        f"({cv_difference:.4f} difference)."
    )

Threshold 0.20 improved mean CV F1 by 0.0096.


### Threshold Result

The `0.20` threshold improved the mean CV F1 from `0.6579` to `0.6675`.

The standard deviation changed only slightly, from `0.0057` to `0.0060`, so the higher F1 did not come with a large loss in stability.

This is a stronger result than the single validation improvement from Day 1 because the same threshold performs better across all five development folds.

## 17. Try Nearby Thresholds

The threshold `0.20` is already better than the default prediction rule, but in Day 1 I only tried a few values:

```text
0.10, 0.20, 0.30, 0.40, 0.50
```

That leaves a fairly large gap around `0.20`.

I now test a few nearby thresholds from `0.12` to `0.30` using the same five stratified folds. The test set is still not used.

I fit Logistic Regression once per fold, keep its validation probabilities, then calculate F1 for each candidate threshold. This avoids retraining the same model again for every threshold.

In [18]:
candidate_thresholds = [
    round(value / 100, 2)
    for value in range(12, 31)
]

threshold_scores = {
    threshold: []
    for threshold in candidate_thresholds
}

for train_idx, val_idx in stratified_kfold.split(X_dev, y_dev):
    X_fold_train = X_dev.iloc[train_idx]
    X_fold_val = X_dev.iloc[val_idx]
    y_fold_train = y_dev.iloc[train_idx]
    y_fold_val = y_dev.iloc[val_idx]

    fold_model = LogisticRegression(
        max_iter=1000,
        random_state=42,
    )

    fold_model.fit(X_fold_train, y_fold_train)

    fold_probabilities = fold_model.predict_proba(X_fold_val)[:, 1]

    for threshold in candidate_thresholds:
        fold_predictions = (
            fold_probabilities >= threshold
        ).astype(int)

        threshold_scores[threshold].append(
            f1_score(y_fold_val, fold_predictions)
        )

I summarize each threshold using the mean F1 across the five folds and its standard deviation.

The best threshold is selected by **mean CV F1**, not by the score of one fold.

In [19]:
threshold_search_rows = []

for threshold, scores in threshold_scores.items():
    scores = pd.Series(scores)

    threshold_search_rows.append({
        "threshold": threshold,
        "mean_cv_f1": scores.mean(),
        "cv_std": scores.std(ddof=0),
    })

threshold_search_df = (
    pd.DataFrame(threshold_search_rows)
    .sort_values("mean_cv_f1", ascending=False)
    .reset_index(drop=True)
)

threshold_search_df.head(10).round(4)

,threshold,mean_cv_f1,cv_std
0,0.12,0.6679,0.0057
1,0.13,0.6679,0.0057
2,0.14,0.6679,0.0057
3,0.15,0.6679,0.0057
4,0.16,0.6679,0.0057
5,0.17,0.6679,0.0057
6,0.18,0.6679,0.0057
7,0.19,0.6679,0.0057
8,0.20,0.6675,0.0060
9,0.21,0.6667,0.0068


### Nearby Threshold Result

The search shows a very flat region around the value chosen in Day 1.

Thresholds from about `0.12` to `0.19` all round to a mean CV F1 of `0.6679`, while `0.20` gives `0.6675`.

The code selects `0.12` because it has the highest mean CV F1 in the search, but the improvement over `0.20` is only `0.0004`. That difference is much smaller than the fold-to-fold standard deviation, so I do not treat it as a meaningful improvement over `0.20`.

The useful conclusion is that lowering the default threshold clearly helps, and the good threshold region is fairly broad around the value I found in Day 1.

In [20]:
best_threshold = threshold_search_df.loc[0, "threshold"]
best_threshold_mean = threshold_search_df.loc[0, "mean_cv_f1"]
best_threshold_std = threshold_search_df.loc[0, "cv_std"]

improvement_over_default = best_threshold_mean - cv_mean
improvement_over_020 = best_threshold_mean - threshold_cv_mean

print(f"Best threshold: {best_threshold:.2f}")
print(
    f"Best mean CV F1: "
    f"{best_threshold_mean:.4f} ± {best_threshold_std:.4f}"
)
print(
    f"Improvement over default LR: "
    f"{improvement_over_default:.4f}"
)
print(
    f"Improvement over threshold 0.20: "
    f"{improvement_over_020:.4f}"
)

Best threshold: 0.12
Best mean CV F1: 0.6679 ± 0.0057
Improvement over default LR: 0.0100
Improvement over threshold 0.20: 0.0004


## 18. Refit the Highest-Mean Configuration

The search code selects the threshold with the highest mean CV F1 and then fits one Logistic Regression model on the full development set.

This refit uses all development samples after the threshold comparison is finished.

In [21]:
final_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

final_model.fit(X_dev, y_dev)

selected_threshold = float(best_threshold)

print(
    f"Selected threshold from development CV: "
    f"{selected_threshold:.2f}"
)
print(
    f"Final model trained on "
    f"{len(X_dev)} development samples."
)

Selected threshold from development CV: 0.12
Final model trained on 219761 development samples.


## 19. Test Set Note

The test set is not used in the cross-validation or the nearby-threshold search.

I already evaluated the held-out test set in Day 1, so I do not reuse its score to choose the new threshold.

Because the threshold is selected using the development folds, the CV score here is a development/tuning result, not a new final test estimate.

## 20. Summary

The first cross-validation check confirmed that the default Logistic Regression result was stable:

```text
Day 1 single validation F1 : 0.6567
5-fold default mean F1     : 0.6579
CV standard deviation      : 0.0057
```

Using the `0.20` threshold from Day 1 improved the 5-fold result:

```text
Threshold 0.20 mean CV F1  : 0.6675
Threshold 0.20 CV std      : 0.0060
```

The nearby-threshold search found a best mean CV F1 of `0.6679`. Several thresholds between about `0.12` and `0.19` give almost the same result, so I consider this a flat good region rather than evidence that `0.12` is clearly better than `0.20`.

The important improvement is from the default Logistic Regression (`0.6579`) to the lower-threshold version (about `0.6675–0.6679`). The fold variation stays small, so this improvement is reasonably stable across the development splits.